# Distribution Analysis — GDP per Capita & Unemployment Rate (2025)

Two histograms showing the **distribution** of values using all quarterly observations (146 rows — 38 countries × 4 quarters of 2025).

A histogram divides values into intervals (bins) and counts how many observations fall in each interval. This reveals:
- **Central tendency** — mean and median, and why they differ in skewed distributions
- **Dispersion** — how spread out the values are across countries and quarters
- **Skewness** — GDP is right-skewed (skew ≈ 1.70) driven by Ireland and Luxembourg; unemployment is near-symmetric (skew ≈ 0.48)
- **Outliers** — Ireland ($128K GDP) and Luxembourg are clearly visible in the right tail

Data source: `datasets/gdp_unemployment_merged.csv` (inner join, 38 OECD member countries × 4 quarters)


In [20]:
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Load merged quarterly dataset (Option 1: 146 rows, 38 countries × 4 quarters) ──
df = pd.read_csv('datasets/gdp_unemployment_merged.csv', sep=';')

gdp_vals = df['GDP_PER_CAPITA_USD_PPP'].dropna().values
une_vals = df['UNE_RATE_PCT'].dropna().values

gdp_mean,   gdp_median   = gdp_vals.mean(),   np.median(gdp_vals)
une_mean,   une_median   = une_vals.mean(),   np.median(une_vals)
gdp_std,    une_std      = gdp_vals.std(),    une_vals.std()
gdp_skew = pd.Series(gdp_vals).skew()
une_skew = pd.Series(une_vals).skew()

# ── Combined figure: GDP histogram (top) + Unemployment histogram (bottom) ──
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.55, 0.45],
    subplot_titles=[
        f'GDP per Capita — {len(gdp_vals)} observations ({df["REF_AREA"].nunique()} countries × 4 quarters) | skew = {gdp_skew:.2f}',
        f'Unemployment Rate — {len(une_vals)} observations | skew = {une_skew:.2f}',
    ],
    vertical_spacing=0.14,
)

# GDP histogram
fig.add_trace(go.Histogram(
    x=gdp_vals,
    nbinsx=12,
    marker_color='#1f77b4',
    marker_line=dict(color='white', width=1),
    name='GDP',
    hovertemplate='GDP: $%{x:,.0f}<br>Count: %{y}<extra></extra>',
), row=1, col=1)

# Unemployment histogram
fig.add_trace(go.Histogram(
    x=une_vals,
    nbinsx=12,
    marker_color='#2ca02c',
    marker_line=dict(color='white', width=1),
    name='Unemployment',
    hovertemplate='Rate: %{x:.2f}%<br>Count: %{y}<extra></extra>',
), row=2, col=1)

# Mean / Median / ±1 SD lines — GDP
for x_val, dash, color, label in [
    (gdp_mean,          'solid', '#d62728', f'Mean ${gdp_mean:,.0f}'),
    (gdp_median,        'dot',   '#9467bd', f'Median ${gdp_median:,.0f}'),
    (gdp_mean - gdp_std,'dash',  '#aec7e8', f'-1 SD ${gdp_mean-gdp_std:,.0f}'),
    (gdp_mean + gdp_std,'dash',  '#aec7e8', f'+1 SD ${gdp_mean+gdp_std:,.0f}'),
]:
    fig.add_vline(x=x_val, line=dict(color=color, width=1.5, dash=dash), row=1, col=1)
    fig.add_annotation(
        x=x_val, y=1.08, xref='x', yref='y domain',
        text=label, showarrow=False,
        font=dict(color=color, size=9), xanchor='center', row=1, col=1
    )

# Mean / Median / ±1 SD lines — Unemployment
for x_val, dash, color, label in [
    (une_mean,          'solid', '#d62728', f'Mean {une_mean:.2f}%'),
    (une_median,        'dot',   '#9467bd', f'Median {une_median:.2f}%'),
    (une_mean - une_std,'dash',  '#98df8a', f'-1 SD {une_mean-une_std:.2f}%'),
    (une_mean + une_std,'dash',  '#98df8a', f'+1 SD {une_mean+une_std:.2f}%'),
]:
    fig.add_vline(x=x_val, line=dict(color=color, width=1.5, dash=dash), row=2, col=1)
    fig.add_annotation(
        x=x_val, y=1.08, xref='x2', yref='y2 domain',
        text=label, showarrow=False,
        font=dict(color=color, size=9), xanchor='center', row=2, col=1
    )

fig.update_layout(
    title_text='OECD Countries — GDP per Capita & Unemployment Rate Distribution (2025, quarterly)',
    height=680,
    margin=dict(t=80, b=40),
    plot_bgcolor='white',
    showlegend=False,
    bargap=0.05,
)
fig.update_xaxes(gridcolor='#e0e0e0', row=1, col=1, title_text='GDP per Capita (USD PPP, constant 2020)')
fig.update_xaxes(gridcolor='#e0e0e0', row=2, col=1, title_text='Unemployment Rate (% of labour force)')
fig.update_yaxes(gridcolor='#e0e0e0', title_text='Observations (country-quarters)', dtick=2)

fig.show()